# CuPyCCx — GPU test on Google Colab

**Before running:** set runtime to GPU via *Runtime → Change runtime type → T4 GPU*

In [ ]:
# Cell 1 — confirm GPU is available
!nvidia-smi
!nvcc --version

In [ ]:
%%bash
# Cell 2 — install system and Python dependencies
apt-get install -qq cmake ninja-build libeigen3-dev libopenblas-dev
pip install -q --upgrade pip
pip install -q scikit-build-core pybind11 pyscf

In [ ]:
%%bash
# Cell 3 — clone and build with CUDA (T4 = sm_75; A100 = sm_80)
git clone --quiet https://github.com/varunrishi/CuPyCCx.git
cd CuPyCCx
pip install -q -e . \
  -C cmake.define.CUPYCCX_CUDA=ON \
  -C cmake.define.CUPYCCX_CUDA_ARCH=75 \
  -C cmake.build-type=Release
echo 'Build complete'

In [ ]:
import os, time
os.chdir('CuPyCCx')

from pyscf import gto, scf
from cupyccx.scf_data import prepare_from_pyscf
from cupyccx.method import CCD, CCOptions

# Cell 4 — N2/STO-3G: CPU vs GPU correctness check
mol  = gto.M(atom='N 0 0 0; N 0 0 2.118', basis='sto-3g', unit='Bohr', verbose=0)
mf   = scf.RHF(mol).run()
data = prepare_from_pyscf(mf, verbose=False)

opts_cpu = CCOptions(use_gpu=False, max_iter=200, conv_energy=1e-9, conv_amp=1e-8)
opts_gpu = CCOptions(use_gpu=True,  max_iter=200, conv_energy=1e-9, conv_amp=1e-8)

t0 = time.time()
r_cpu = CCD.from_scf_data(data, opts=opts_cpu).compute(e_scf=data.e_scf)
t_cpu = time.time() - t0

t0 = time.time()
r_gpu = CCD.from_scf_data(data, opts=opts_gpu).compute(e_scf=data.e_scf)
t_gpu = time.time() - t0

print(f'CPU  E_corr = {r_cpu.e_corr:.12f} Ha  ({t_cpu:.2f}s)')
print(f'GPU  E_corr = {r_gpu.e_corr:.12f} Ha  ({t_gpu:.2f}s)')
print(f'Diff        = {abs(r_cpu.e_corr - r_gpu.e_corr):.2e} Ha')

In [ ]:
# Cell 5 — larger system (cc-pVDZ) to see GPU speedup
mol2  = gto.M(atom='N 0 0 0; N 0 0 2.118', basis='cc-pVDZ', unit='Bohr', verbose=0)
mf2   = scf.RHF(mol2).run()
data2 = prepare_from_pyscf(mf2, verbose=False)
print(f'n_occ={data2.n_occ}  n_vir={data2.n_vir}  n_mo={data2.n_mo}')

opts_cpu = CCOptions(use_gpu=False, max_iter=200, conv_energy=1e-9, conv_amp=1e-8)
opts_gpu = CCOptions(use_gpu=True,  max_iter=200, conv_energy=1e-9, conv_amp=1e-8)

t0 = time.time()
r2_cpu = CCD.from_scf_data(data2, opts=opts_cpu).compute(e_scf=data2.e_scf)
t_cpu = time.time() - t0

t0 = time.time()
r2_gpu = CCD.from_scf_data(data2, opts=opts_gpu).compute(e_scf=data2.e_scf)
t_gpu = time.time() - t0

print(f'CPU  E_corr = {r2_cpu.e_corr:.12f} Ha  ({t_cpu:.2f}s)')
print(f'GPU  E_corr = {r2_gpu.e_corr:.12f} Ha  ({t_gpu:.2f}s)')
print(f'Speedup     = {t_cpu/t_gpu:.1f}x')